In [61]:
import numpy as np
import os
import time
from pydrake.all import (
    Meshcat,
    Simulator,
    RigidTransform,
    RollPitchYaw,
    SceneGraphCollisionChecker,
    Context,
    TrajectorySource,
	PiecewisePolynomial,
	BasicVector,
	LeafSystem,
	DiagramBuilder,
)
from pydrake.multibody.plant import MultibodyPlant
from manipulation.station import LoadScenario, MakeHardwareStation

import sys
sys.path.append("../src")
from ik import IKSolver
from rrt import RRTTools
from motion_planning import *

In [62]:
meshcat = Meshcat()

INFO:drake:Meshcat listening for connections at http://localhost:7020


In [63]:
# Set up paths
scenario_file = os.path.abspath("../kitchen_model/real_kitchen_scenario.yaml")
kitchen_model_path = os.path.abspath("../kitchen_model")
assets_path = os.path.abspath("../assets")

# Read the scenario YAML and replace placeholders
with open(scenario_file, 'r') as f:
    scenario_data = f.read()

scenario_data = scenario_data.replace("{KITCHEN_MODEL_PATH}", kitchen_model_path)
scenario_data = scenario_data.replace("{ASSETS_PATH}", assets_path)

# Load the scenario
scenario = LoadScenario(data=scenario_data)
station = MakeHardwareStation(scenario)
sim = Simulator(station)
context = sim.get_mutable_context()

In [64]:
plant = station.GetSubsystemByName("plant")
plant_context = plant.GetMyMutableContextFromRoot(context)

# Get mobile base indices
ix = plant.GetJointByName("iiwa_base_x").position_start()
iy = plant.GetJointByName("iiwa_base_y").position_start()
iz = plant.GetJointByName("iiwa_base_z").position_start()
j1 = plant.GetJointByName("iiwa_joint_1").position_start()

# Get gripper frame
gripper_body = plant.GetBodyByName("body")

# Get initial configuration and limits
active_lo, active_hi = 3, 10
q0 = plant.GetPositions(plant_context).copy()
qlo = plant.GetPositionLowerLimits()
qhi = plant.GetPositionUpperLimits()
vlo = plant.GetVelocityLowerLimits()
vhi = plant.GetVelocityUpperLimits()

print(f"Initial configuration: {q0[3:10]}")
print(f"Initial (x,y,z) position: ({q0[ix]:.2f}, {q0[iy]:.2f}, {q0[iz]:.2f})")
qlo[ix], qhi[ix] = -8, -4
qlo[iy], qhi[iy] = -4, 0
qlo[j1], qhi[j1] = -0.01, 0.01

Initial configuration: [ 0.   0.1  0.  -1.2  0.   1.6  0. ]
Initial (x,y,z) position: (-4.00, -1.00, 0.00)


## IK for Grasp Pose and Place Poses

In [ ]:
# Get current end-effector pose for comparison
plant.SetPositions(plant_context, q0)
q0new = plant.GetPositions(plant_context).copy()
X_WG_current = plant.EvalBodyPoseInWorld(plant_context, gripper_body)
print(f"Current base position: ({q0new[ix]:.2f}, {q0new[iy]:.2f}, {q0new[iz]:.2f})")
print(f"Current gripper position: {X_WG_current.translation()}")

# Define target gripper pose in world frame
target_position = np.array([-4.4, -1.525, 0.925])
target_rpy = RollPitchYaw([-np.pi/12, 0, np.pi/2])
X_WG_grasp = RigidTransform(target_rpy, target_position)
print(f"Target gripper position: {target_position}")

# Define target place pose in world frame
target_position = np.array([-4.4, -1.9, 0.925])
target_rpy = RollPitchYaw([-np.pi/12, 0, np.pi/2])
X_WG_place = RigidTransform(target_rpy, target_position)
print(f"Target gripper position: {target_position}")

SystemExit: Failure at bazel-out/darwin_arm64-opt/bin/external/drake+/multibody/plant/_virtual_includes/multibody_plant_core/drake/multibody/plant/multibody_plant.h:2894 in SetPositions(): condition 'q.size() == num_positions()' failed.

/Users/seyoungree/locomanipulation-4212-final-project/.venv/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [109]:
collision_checker = SceneGraphCollisionChecker(
						model=station,
						robot_model_instances=[station.plant().GetModelInstanceByName("mobile_iiwa")],
						edge_step_size=0.01,
						env_collision_padding=0.01,
						self_collision_padding=0.01,)

INFO:drake:Allocating contexts to support implicit context parallelism 8


In [110]:
q0new = q0.copy()
# q0new[iy] = -1.5
q0new[iz] = 0.03
q0new[3:10] = 0
q0new[0:2] = [-4.5, -2.92]
plant.SetPositions(plant_context, q0new)
collision_checker.CheckConfigCollisionFree(q0new)

True

In [117]:
ik_solver = IKSolver(
    plant=plant,
    plant_context=plant_context,
    ix=ix,
    iy=iy,
    iz=iz,
    active_hi=active_hi,
    q0=q0,
    gripper_body=gripper_body,
	theta_bound=0.2 * np.pi,
	pos_tol=0.04,
	base_tol=0.1,
	col_dist_tol=0.01
	
)

init_guess = q0.copy()
init_guess[0:2] = [-4.5, -2.92]
q_grasp = ik_solver.solve(X_WG_grasp, init_guess)

In [134]:
q_grasp

array([-4.4004417 , -2.91778252,  0.02935388,  0.03622381,  0.03685292,
       -0.18278937, -0.80933222, -1.36926305,  1.94631227, -1.89012708,
        0.        ,  0.        ,  1.        ,  0.        ,  0.        ,
        0.        , -4.5       , -1.5       ,  0.95      ,  1.        ,
        0.        ,  0.        ,  0.        , -5.3       , -2.        ,
        0.95      ,  1.        ,  0.        ,  0.        ,  0.        ,
       -5.7       , -2.        ,  0.95      ,  1.        ,  0.        ,
        0.        ,  0.        , -5.4       , -3.15      ,  0.95      ,
        1.        ,  0.        ,  0.        ,  0.        , -4.3       ,
       -3.15      ,  0.95      ])

In [118]:
collision_checker.CheckConfigCollisionFree(q_grasp)

True

In [119]:
q_upright = q_grasp.copy()
q_upright[active_lo:active_hi] = 0

In [120]:
plant.SetPositions(plant_context, q0new)

## Pick

In [121]:
tools = RRTTools(
        collision_checker=collision_checker,
        q_lo=qlo, q_hi=qhi,
        df_start=active_lo, df_end=active_hi,
        step_size=0.01,
        goal_threshold=0.01,
    )

rrt_path_pick = tools.plan(q_start=q0new, q_goal=q_grasp, max_iterations=50_000)

[RRT] iteration 0
[RRT] Connected in 1 iterations


In [122]:
rrt_path_upright = tools.plan(q_start=q_grasp, q_goal=q_upright, max_iterations=50_000)

[RRT] iteration 0
[RRT] Connected in 1 iterations


In [123]:
# visualize_rrt_waypoints(rrt_path_pick, station, meshcat)

In [124]:
shortcutted_path_pick = tools.shortcut_path(rrt_path_pick, passes=200, min_separation=2, max_step=0.1)

In [125]:
shortcutted_path_upright = tools.shortcut_path(rrt_path_upright, passes=200, min_separation=2, max_step=0.1)

In [126]:
shortcutted_path_pick = [cut_path[:10] for cut_path in shortcutted_path_pick]
shortcutted_path_upright = [cut_path[:10] for cut_path in shortcutted_path_upright]

traj_q_pick, traj_wsg_pick = build_trajs_pick(shortcutted_path_pick, shortcutted_path_upright, q_grasp[:10])

[build_trajs_pick] q_samples shape: (10, 69), T=6.250s


## Place

In [127]:
q_upright_place = q_upright.copy()
q_place = q_grasp.copy()
rrt_path_place = tools.plan(q_start=q_upright_place, q_goal=q_place, max_iterations=50_000)
shortcutted_path_place = tools.shortcut_path(rrt_path_place, passes=200, min_separation=2, max_step=0.1)

[RRT] iteration 0
[RRT] Connected in 1 iterations


In [128]:
rrt_path_place_upright = tools.plan(q_start=q_place, q_goal=q_upright, max_iterations=50_000)
shortcutted_path_place_upright = tools.shortcut_path(rrt_path_place_upright, passes=200, min_separation=2, max_step=0.1)

[RRT] iteration 0
[RRT] Connected in 1 iterations


In [129]:
shortcutted_path_place = [cut_path[:10] for cut_path in shortcutted_path_pick]
shortcutted_path_place_upright = [cut_path[:10] for cut_path in shortcutted_path_upright]

traj_q_place, traj_wsg_place = build_trajs_place(shortcutted_path_place, shortcutted_path_place_upright, q_grasp[:10], traj_q_pick.end_time())

[build_trajs_place] q_samples shape: (10, 68), T=11.500s


In [130]:
traj_q_pick.ConcatenateInTime(traj_q_place)
traj_wsg_pick.ConcatenateInTime(traj_wsg_place)

In [131]:
model_drivers = """
model_drivers:
  wsg: !SchunkWsgDriver {}
  mobile_iiwa: !InverseDynamicsDriver {}
"""

scenario_data_w_driver = scenario_data + model_drivers

In [132]:
class IiwaDesiredStateFromQ(LeafSystem):
    def __init__(self, nq, nv):
        super().__init__()
        self.nq = nq
        self.nv = nv
        self.DeclareVectorInputPort("q_des", BasicVector(nq))
        self.DeclareVectorOutputPort(
            "x_des", BasicVector(nq + nv), self.CalcOutput
        )

    def CalcOutput(self, context: Context, output: BasicVector):
        q_des = self.get_input_port(0).Eval(context)
        x_des = np.zeros(self.nq + self.nv)
        x_des[:self.nq] = q_des
        output.SetFromVector(x_des)

meshcat = Meshcat()
scenario = LoadScenario(data=scenario_data_w_driver)
builder = DiagramBuilder()
station = builder.AddSystem(MakeHardwareStation(scenario, meshcat=meshcat))

plant = station.GetSubsystemByName("plant")
mobile_iiwa = plant.GetModelInstanceByName("mobile_iiwa")

nq = plant.num_positions(mobile_iiwa)
nv = plant.num_velocities(mobile_iiwa)

# Trajectory sources
traj_source_q = builder.AddSystem(TrajectorySource(traj_q_pick))
traj_source_wsg = builder.AddSystem(TrajectorySource(traj_wsg_pick))
q_to_x = builder.AddSystem(IiwaDesiredStateFromQ(nq, nv))

builder.Connect(
    traj_source_q.get_output_port(),
    q_to_x.get_input_port(0),
)

# Feed desired state into the InverseDynamicsDriver
builder.Connect(
    q_to_x.get_output_port(),
    station.GetInputPort("mobile_iiwa.desired_state"),
)

builder.Connect(
    traj_source_wsg.get_output_port(),
    station.GetInputPort("wsg.position"),
)

diagram = builder.Build()
simulation = Simulator(diagram)
simulation.set_target_realtime_rate(1.0)

ctx = simulation.get_mutable_context()
plant_context = plant.GetMyMutableContextFromRoot(ctx)

# Make the physical plant start at first waypoint of traj_q
q0 = traj_q_pick.value(0.0).flatten()
plant.SetPositions(plant_context, mobile_iiwa, q0)
plant.SetVelocities(plant_context, mobile_iiwa, np.zeros(nv))

diagram.ForcedPublish(ctx)
meshcat.StartRecording()
simulation.Initialize()
simulation.AdvanceTo(max(traj_q_pick.end_time(), traj_wsg_pick.end_time()))
meshcat.StopRecording()
meshcat.PublishRecording()

INFO:drake:Meshcat listening for connections at http://localhost:7001
